# CB Loss — Class-Balanced Loss from the Effective Number of Samples (2019)

_Papers · Computer Vision_

**Reweight any loss by 1/(effective number of samples) per class, so a long tail of rare classes stops being ignored — without the instability of raw inverse-frequency weighting.**

---

This notebook is a practice scaffold for the **CB Loss — Class-Balanced Loss from the Effective Number of Samples (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. Effective-number class weights (Eqn 2 + Sec 4 of arXiv:1901.05555).
#    E_n = (1 - beta^n)/(1 - beta);  w_y = 1/E_{n_y}, normalized to sum C.
# ============================================================
def cb_weights(counts, beta, num_classes=None, normalize=True):
    counts = torch.tensor(counts, dtype=torch.float64)
    eff_num = (1.0 - torch.pow(beta, counts)) / (1.0 - beta) if beta > 0 else torch.ones_like(counts)
    w = 1.0 / eff_num
    if normalize:
        C = num_classes or len(counts)
        w = w * C / w.sum()
    return w

counts = [5000, 1000, 200, 50, 10]
for b in [0.0, 0.9, 0.99, 0.999, 0.9999]:
    w = cb_weights(counts, b)
    print(f"beta={b:<7}: weights={[round(x,3) for x in w.tolist()]}")
# beta=0.0    : weights=[1.0, 1.0, 1.0, 1.0, 1.0]            <- no re-weighting
# beta=0.99   : weights=[0.31, 0.31, 0.358, 0.784, 3.239]   <- head SATURATES to a shared floor
# beta=0.9999 : weights=[0.01, 0.041, 0.2, 0.793, 3.956]    <- approaches inverse-frequency

# ---- Sanity: the two limits ----
inv_freq = cb_weights(counts, 1 - 1e-12)                     # beta -> 1
manual   = torch.tensor([1.0/n for n in counts], dtype=torch.float64)
manual   = manual * len(counts) / manual.sum()
print("beta->1 == inverse-frequency:", torch.allclose(inv_freq, manual, atol=1e-4))  # True
print("beta=0  all-equal           :", torch.allclose(cb_weights(counts, 0.0),
                                                       torch.ones(len(counts), dtype=torch.float64)))  # True

# ============================================================
# 2. Class-balanced focal loss (Eqn 6). gamma=0 -> class-balanced cross entropy.
# ============================================================
def cb_focal_loss(logits, targets, counts, beta=0.999, gamma=2.0):
    w = cb_weights(counts, beta, logits.shape[1]).to(logits.dtype)   # (C,)
    logp = F.log_softmax(logits, dim=1)                              # (B,C)
    pt   = logp.gather(1, targets[:, None]).squeeze(1).exp()         # prob of true class
    ce   = -logp.gather(1, targets[:, None]).squeeze(1)             # per-example CE
    focal = (1 - pt) ** gamma * ce                                   # focal (paper-retinanet)
    return (w[targets] * focal).mean()                              # class-balanced multiply

# ---- Sanity: gamma=0 CB-focal == weight-applied cross entropy ----
torch.manual_seed(0)
z = torch.randn(32, 5); y = torch.randint(0, 5, (32,))
a = cb_focal_loss(z, y, counts, beta=0.999, gamma=0.0)
w = cb_weights(counts, 0.999, 5).float()
b = (w[y] * F.cross_entropy(z, y, reduction="none")).mean()
print("CB-focal(gamma=0) == CB cross entropy:", torch.allclose(a, b, atol=1e-6))  # True

# ============================================================
# 3. Train a long-tailed 5-class classifier: plain CE vs CB-focal.
#    Counts [400,200,80,30,10]; balanced test set -> read TAIL recall.
# ============================================================
g = torch.Generator().manual_seed(1)
centers = torch.tensor([[2.,2.],[ -2.,2.],[2.,-2.],[-2.,-2.],[0.,0.]])
tr_counts = [400, 200, 80, 30, 10]
Xtr = torch.cat([centers[c] + 1.4*torch.randn(n,2,generator=g) for c,n in enumerate(tr_counts)])
ytr = torch.cat([torch.full((n,), c) for c,n in enumerate(tr_counts)])
Xte = torch.cat([centers[c] + 1.4*torch.randn(60,2,generator=g) for c in range(5)])   # balanced test
yte = torch.cat([torch.full((60,), c) for c in range(5)])

def train(kind, beta=0.999, gamma=2.0, steps=600):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32,5))
    opt = torch.optim.Adam(net.parameters(), lr=0.03)
    for _ in range(steps):
        opt.zero_grad()
        logits = net(Xtr)
        loss = (F.cross_entropy(logits, ytr) if kind=="ce"
                else cb_focal_loss(logits, ytr, tr_counts, beta=beta, gamma=gamma))
        loss.backward(); opt.step()
    with torch.no_grad():
        pred = net(Xte).argmax(1)
        per_class = [ (pred[yte==c]==c).float().mean().item() for c in range(5) ]
    return per_class

ce  = train("ce")
cb  = train("cbf")
print("per-class recall  CE     :", [round(x,2) for x in ce])
print("per-class recall  CB-focal:", [round(x,2) for x in cb])
print("balanced acc  CE / CB-focal:", round(sum(ce)/5,3), "/", round(sum(cb)/5,3))
# CE      : tail (rarest) recall is low; head is high.
# CB-focal: tail recall rises, balanced accuracy improves.
# Our small run, not the paper's reported numbers.

# ============================================================
# 4. Ablation over beta (gamma fixed) -> balanced accuracy.
# ============================================================
for b in [0.0, 0.9, 0.99, 0.999, 0.9999]:
    pc = train("cbf", beta=b)
    print(f"beta={b:<7}: balanced_acc={round(sum(pc)/5,3)}  tail_recall={round(pc[-1],2)}")
# beta=0.0 behaves like CE; mid-beta lifts the tail; beta->1 can over-weight the tail.
# Our small-scale run, not the paper's reported numbers.

## Visualize the data & results

_How does the 'effective number' of samples saturate as you add data, and how does the resulting class weight sit between 'treat all classes equally' and 'inverse frequency'?_

In [ ]:
import torch
# Effective number and class weights, computed exactly from the paper's formulas.
def cb_weights(counts, beta):
    counts = torch.tensor(counts, dtype=torch.float64)
    eff = (1 - torch.pow(beta, counts)) / (1 - beta) if beta > 0 else torch.ones_like(counts)
    w = 1 / eff
    return (w * len(counts) / w.sum()).tolist()          # normalize to sum = C

counts = [5000, 1000, 200, 50, 10]
print("no re-weighting :", [1]*5)
print("CB beta=0.99    :", [round(x,3) for x in cb_weights(counts, 0.99)])
print("inverse-freq    :", [round(x,3) for x in cb_weights(counts, 1 - 1e-12)])
# no re-weighting : [1, 1, 1, 1, 1]
# CB beta=0.99    : [0.31, 0.31, 0.358, 0.784, 3.239]
# inverse-freq    : [0.008, 0.04, 0.198, 0.792, 3.962]
# Computed from (1-beta^n)/(1-beta); not the paper's reported numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With $\beta=0.99$, compute the effective number $E_n$ for a class with $n=100$ samples, and its ceiling.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ceiling is $1/(1-\beta)=1/0.01=100$. — _As $n\to\infty$, $\beta^n\to0$ so $E_n\to 1/(1-\beta)$._
- $E_{100}=(1-0.99^{100})/0.01$. Since $0.99^{100}\approx0.366$, $E_{100}\approx(1-0.366)/0.01=63.4$. — _$0.99^{100}=e^{100\ln 0.99}=e^{-1.005}\approx0.366$._

**Answer:** $E_{100}\approx 63.4$ effective samples, ceiling $100$ — already most of the way to saturation, so a class with 100 samples is treated almost like a much larger one.

</details>

**Problem 2.** Counts are $[5000,1000,200,50,10]$. Explain why, at $\beta=0.99$, the two largest classes end up with the SAME weight, but under inverse-frequency they do not.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At $\beta=0.99$ the effective-number ceiling is 100; both $n=5000$ and $n=1000$ give $E_n\approx100$ (both far past saturation). — _$0.99^{1000}$ and $0.99^{5000}$ are both essentially 0, so $E_n\approx1/(1-\beta)=100$ for each._
- So $w=1/E$ is the same for both; inverse-frequency uses $1/n$, which is $1/5000$ vs $1/1000$ — a $5\times$ difference. — _Effective number saturates; raw count does not._

**Answer:** CB saturates the head to a shared floor (here ~0.31 after normalizing), so 5000 and 1000 look equally "covered"; inverse-frequency keeps a $5\times$ gap because it never saturates. That saturation is exactly the stability CB adds over $1/n$.

</details>

**Problem 3.** ABLATION. You set $\beta=0.9999$ on a set whose rarest class has 5 samples and the data is noisy. What failure do you expect, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At $\beta=0.9999$, $E_5=(1-0.9999^5)/0.0001\approx(1-0.9995)/0.0001\approx5.0$, so $w\approx1/5$ — essentially inverse-frequency. — _Large $\beta$ pushes $E_n\to n$, recovering $1/n_y$._
- That rarest class now pulls ~$1000\times$ harder than a 5000-sample head class. — _$w_{\text{tail}}/w_{\text{head}} \approx (1/5)/(1/5000)=1000$._

**Answer:** Near-inverse-frequency over-weighting: a handful of noisy tail examples dominate the gradient, so head accuracy can collapse and training gets unstable — the exact regime CB is meant to avoid by choosing a smaller $\beta$ so the head saturates instead.

</details>